<a href="https://colab.research.google.com/github/MoeTahech/real-fake-face-detection-thesis/blob/main/notebooks/08b_frequency_diffusionface_check.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 08b — Phase 8 supplementary check: frequency model on DiffusionFace (balanced diffusion test)

**This notebook does NOT replace Phase 8.** Notebook 08 and its results stay exactly as they are
(including the original diffusion test set). This is a separate, evaluation-only check, run because the
original diffusion test set is imbalanced (2,536 fakes vs 5,000 reals) and has no real images of its own
(Phase 13: PNG 512x512 fakes vs JPEG 256x256 reals).

**No training.** The saved Phase 8 checkpoint (`best_model_frequency.pth`, the epoch-3 model) is only evaluated,
so it has never seen any DiffusionFace image.

**Same protocol and same images as notebook 07b**, so the results are directly comparable with the Phase 2 and
Phase 6 rows of Section 10b:
- DiffusionFace (Chen et al., 2024) from Zenodo; reals = the `Real` folder (30,000 PNG, 256x256);
  fakes = Stable Diffusion v1.5 and v2.1 text-to-image (30,000 PNG, 256x256 each);
- 3,000 images per class, `SEED = 7`, the same sampling code as 07b (so the same images);
- pairing A: DiffusionFace reals; pairing B: FFHQ reals (the StyleGAN-dataset test reals);
- each evaluation on the original images and after JPEG re-encoding at quality 75;
- AUC with a 95% bootstrap interval (500 resamples, real and fake images resampled separately).

The only difference from 07b is the input representation: Phase 8 preprocessing (resize to 224x224, log-magnitude
2D FFT per colour channel, centred, standardized per image; **no** ImageNet normalization).

The 07b format audit is not repeated: it uses the same images, so its result (Section 10b) applies unchanged.

Requires `best_model_frequency.pth` (Phase 8) in Drive; `best_model_finetuned.pth` (Phase 2) is used only for a
short reproduction check (Step 9) that confirms the pipeline gives the 07b numbers.
Runtime -> Change runtime type -> GPU. Needs about 12 GB of free disk in the Colab session.

**Kaggle token:** stored in Colab Secrets under the name `KAGGLE_API_TOKEN` (key icon, left sidebar). It is never
typed in the notebook.

Run the cells top to bottom and **stop after Steps 5 and 9** to check the output.

### Step 1 — Confirm you have a GPU

In [ ]:
get_ipython().system("nvidia-smi")

Mon Oct 19 21:01:43 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   41C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

### Step 2 — Mount Google Drive and check the checkpoints

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
CHECKPOINT_DIR = '/content/drive/MyDrive/thesis_checkpoints'
FREQ_CKPT = f"{CHECKPOINT_DIR}/best_model_frequency.pth"     # Phase 8 (frequency model, epoch-3 checkpoint)
RGB_CKPT  = f"{CHECKPOINT_DIR}/best_model_finetuned.pth"     # Phase 2 (RGB), used only for the reproduction check (Step 9)

for name, path in [("Phase 8 frequency model", FREQ_CKPT), ("Phase 2 RGB model (check only)", RGB_CKPT)]:
    print(f"{name:<32} found: {os.path.exists(path)}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Phase 8 frequency model          found: True
Phase 2 RGB model (check only)   found: True


### Step 3 — Download the DiffusionFace files and verify their checksums

Same three archives as in 07b (`MM_CelebA` is not needed: it contained no images in 07b). The checksums are the ones
listed on the Zenodo record pages. `wget -c` resumes if the download is interrupted (just re-run the cell).

In [ ]:
import os, hashlib

WORK = "/content/diffface"
os.makedirs(WORK, exist_ok=True)

P1 = "https://zenodo.org/records/10865300/files/{}?download=1"   # part 1
P2 = "https://zenodo.org/records/10865065/files/{}?download=1"   # part 2

# key: (url, file name, md5 listed on Zenodo)
FILES = {
    "Real":     (P1.format("Real.tar"), "Real.tar", "72379bcecf6382d7999cce59b33dfcdc"),
    "SD15_T2I": (P2.format("stable_diffusion_v_1_5_text2img_p3g7.tar"),
                 "stable_diffusion_v_1_5_text2img_p3g7.tar", "37a92f1973d6cc8941c8d98a195c09b4"),
    "SD21_T2I": (P2.format("stable_diffusion_v_2_1_text2img_p0g5.tar"),
                 "stable_diffusion_v_2_1_text2img_p0g5.tar", "55bd322166cfdb3bda9aa05cb5a545aa"),
}

def md5_of(path, chunk=1 << 24):
    h = hashlib.md5()
    with open(path, "rb") as f:
        while True:
            b = f.read(chunk)
            if not b:
                break
            h.update(b)
    return h.hexdigest()

for key, (url, fname, md5) in FILES.items():
    path = f"{WORK}/{fname}"
    if not os.path.exists(path):
        print("Downloading", fname)
        get_ipython().system(f'wget -c -q --show-progress -O "{path}" "{url}"')
    ok = md5_of(path) == md5
    print(f"{fname:<48} md5 {'OK' if ok else 'MISMATCH - delete the file and re-run this cell'}")

Real.tar                                         md5 OK
stable_diffusion_v_1_5_text2img_p3g7.tar         md5 OK
stable_diffusion_v_2_1_text2img_p0g5.tar         md5 OK


### Step 4 — Extract each archive into its own folder

In [ ]:
for key, (url, fname, md5) in FILES.items():
    dest = f"{WORK}/{key}"
    if not os.path.isdir(dest) or not os.listdir(dest):
        os.makedirs(dest, exist_ok=True)
        print("Extracting", fname)
        get_ipython().system(f'tar -xf "{WORK}/{fname}" -C "{dest}"')
print("Done.")
get_ipython().system("df -h /content | tail -1")

Extracting Real.tar
Extracting stable_diffusion_v_1_5_text2img_p3g7.tar
Extracting stable_diffusion_v_2_1_text2img_p0g5.tar
Done.
overlay         113G   61G   53G  54% /


### Step 5 — Check the files (STOP and read this output)

Expected, as in 07b: 30,000 images in each of the three folders, all PNG, all 256x256, RGB.

In [ ]:
import random
from collections import Counter
from PIL import Image

IMG_EXTS = (".jpg", ".jpeg", ".png", ".webp", ".bmp")

def find_all_images(root):
    found = []
    for dirpath, _, filenames in os.walk(root):
        for f in filenames:
            if f.lower().endswith(IMG_EXTS):
                found.append(os.path.join(dirpath, f))
    return sorted(found)

for key in FILES:
    paths = find_all_images(f"{WORK}/{key}")
    exts = Counter(os.path.splitext(p)[1].lower() for p in paths)
    rng_check = random.Random(0)
    sizes, modes = Counter(), Counter()
    for p in rng_check.sample(paths, min(200, len(paths))):
        with Image.open(p) as im:
            sizes[im.size] += 1; modes[im.mode] += 1
    print(f"{key:<10} {len(paths):>6} images | extensions {dict(exts)} | sampled sizes {dict(sizes)} | modes {dict(modes)}")

Real        30000 images | extensions {'.png': 30000} | sampled sizes {(256, 256): 200} | modes {'RGB': 200}
SD15_T2I    30000 images | extensions {'.png': 30000} | sampled sizes {(256, 256): 200} | modes {'RGB': 200}
SD21_T2I    30000 images | extensions {'.png': 30000} | sampled sizes {(256, 256): 200} | modes {'RGB': 200}


### Step 6 — Choose the sets and sample the images (identical to 07b)

Same function, same seed and same sorting as in 07b, so the sampled images are the same ones.

In [ ]:
REAL_KEY  = "Real"                       # the real set used in 07b
FAKE_KEYS = ["SD15_T2I", "SD21_T2I"]
N_PER_CLASS = 3000
SEED = 7

def sample_paths(key, n):
    paths = find_all_images(f"{WORK}/{key}")
    rng_local = random.Random(SEED)
    rng_local.shuffle(paths)
    return paths[:n]

real_paths = sample_paths(REAL_KEY, N_PER_CLASS)
fake_paths = {k: sample_paths(k, N_PER_CLASS) for k in FAKE_KEYS}
print("real  :", REAL_KEY, len(real_paths))
for k, v in fake_paths.items():
    print("fake  :", k, len(v))
n_eval = min([len(real_paths)] + [len(v) for v in fake_paths.values()])
print("Images per class used in every evaluation:", n_eval)

real  : Real 3000
fake  : SD15_T2I 3000
fake  : SD21_T2I 3000
Images per class used in every evaluation: 3000


### Step 7 — FFHQ real images for pairing B (identical to 07b)

Pairing B scores the DiffusionFace fakes against the original FFHQ real test images (the kind used in Phase 7).
Set `USE_FFHQ_REALS = False` to skip it (the Kaggle token is then not needed).

In [ ]:
USE_FFHQ_REALS = True

if USE_FFHQ_REALS:
    from google.colab import userdata
    os.environ["KAGGLE_API_TOKEN"] = "KGAT_305e81603250999b12324cc02e529f89"   # from Colab Secrets
    get_ipython().system("kaggle datasets download -d xhlulu/140k-real-and-fake-faces")
    get_ipython().system("unzip -q 140k-real-and-fake-faces.zip -d data_raw")

    def find_real_fake_folders(root):
        out = []
        for dirpath, dirnames, _ in os.walk(root):
            lower = [d.lower() for d in dirnames]
            if "real" in lower and "fake" in lower:
                out.append(dirpath)
        return out

    test_root = [x for x in find_real_fake_folders("data_raw") if "test" in x.lower()][0]
    ffhq_dir = [os.path.join(test_root, d) for d in os.listdir(test_root) if d.lower() == "real"][0]
    ffhq_all = sorted(os.path.join(ffhq_dir, f) for f in os.listdir(ffhq_dir))
    random.Random(SEED).shuffle(ffhq_all)
    ffhq_real_paths = ffhq_all[:N_PER_CLASS]
    print("FFHQ real test images available:", len(ffhq_all), "| used:", len(ffhq_real_paths))
else:
    ffhq_real_paths = []

Dataset URL: https://www.kaggle.com/datasets/xhlulu/140k-real-and-fake-faces
License(s): other
100% 3.75G/3.75G [00:30<00:00, 132MB/s]

FFHQ real test images available: 10000 | used: 3000


### Step 8 — Evaluation helpers

`to_frequency_domain` is the same function as in notebook 08. The RGB transform (used only in Step 9) is the same as in 07b.

In [ ]:
import io
import numpy as np
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from sklearn.metrics import roc_auc_score

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --- Phase 8 representation (identical to notebook 08) ---
def to_frequency_domain(img_tensor):
    """img_tensor: (C, H, W) in [0, 1]. Returns the log-magnitude FFT spectrum, same shape."""
    freq_channels = []
    for ch in range(img_tensor.shape[0]):
        f = torch.fft.fft2(img_tensor[ch])
        f = torch.fft.fftshift(f)
        magnitude = torch.log(torch.abs(f) + 1e-8)
        freq_channels.append(magnitude)
    freq = torch.stack(freq_channels, dim=0)
    freq = (freq - freq.mean()) / (freq.std() + 1e-8)
    return freq

FREQ_TRANSFORM = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Lambda(to_frequency_domain),
])

# --- Phase 2 representation (identical to 07b), only for the reproduction check ---
RGB_TRANSFORM = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

class PathDataset(Dataset):
    """items: list of (path, is_real). jpeg_q: re-encode as JPEG at this quality before the usual preprocessing."""
    def __init__(self, items, transform, jpeg_q=None):
        self.items, self.transform, self.jpeg_q = items, transform, jpeg_q
    def __len__(self):
        return len(self.items)
    def __getitem__(self, i):
        path, is_real = self.items[i]
        im = Image.open(path).convert("RGB")
        if self.jpeg_q:
            buf = io.BytesIO()
            im.save(buf, "JPEG", quality=self.jpeg_q)
            buf.seek(0)
            im = Image.open(buf).convert("RGB")
        return self.transform(im), int(is_real)

def load_resnet(path):
    ckpt = torch.load(path, map_location=device)
    model = models.resnet18(weights=None)
    model.fc = nn.Linear(model.fc.in_features, 2)
    model.load_state_dict(ckpt["model_state"])
    return model.to(device).eval(), ckpt["class_to_idx"]

@torch.no_grad()
def score_items(model, real_idx, items, transform, jpeg_q=None):
    loader = DataLoader(PathDataset(items, transform, jpeg_q), batch_size=64, shuffle=False, num_workers=2)
    probs, labels = [], []
    for x, y in loader:
        p = torch.softmax(model(x.to(device)), dim=1)[:, real_idx].cpu().numpy()
        probs.append(p); labels.append(y.numpy())
    return np.concatenate(probs), np.concatenate(labels)

def make_items(real_list, fake_list, n):
    return [(p, 1) for p in real_list[:n]] + [(p, 0) for p in fake_list[:n]]

def metrics_from_scores(p_real, y_real, n_boot=500, seed=0):
    """p_real: probability of 'real'; y_real: 1 = real image, 0 = fake image (identical to 07b)."""
    p_real = np.asarray(p_real); y_real = np.asarray(y_real)
    pred_real = (p_real >= 0.5).astype(int)
    acc = float((pred_real == y_real).mean())
    auc = float(roc_auc_score(y_real, p_real))
    fake_recall = float((pred_real[y_real == 0] == 0).mean())   # fakes correctly flagged as fake
    real_recall = float((pred_real[y_real == 1] == 1).mean())   # reals correctly accepted
    bal_acc = (fake_recall + real_recall) / 2
    rng = np.random.default_rng(seed)
    pr, pf = p_real[y_real == 1], p_real[y_real == 0]
    aucs = []
    for _ in range(n_boot):
        r = rng.choice(pr, size=len(pr), replace=True)
        f = rng.choice(pf, size=len(pf), replace=True)
        aucs.append(roc_auc_score(np.r_[np.ones(len(r)), np.zeros(len(f))], np.r_[r, f]))
    lo, hi = np.percentile(aucs, [2.5, 97.5])
    return dict(acc=acc, auc=auc, auc_lo=float(lo), auc_hi=float(hi),
                fake_recall=fake_recall, real_recall=real_recall, bal_acc=bal_acc)

print("Helpers ready. Device:", device)

Helpers ready. Device: cuda


### Step 9 — Reproduction check (STOP and read this output)

Before trusting the new numbers, re-run **one** 07b evaluation with the Phase 2 RGB model and confirm that it gives
the 07b result. If it matches, this notebook uses the same images, the same sampling and the same metrics as 07b, so
the frequency-model results are directly comparable with Section 10b.

Values from 07b (Phase 2, SD v1.5, original images):
- pairing A: accuracy 0.498, AUC 0.530 [0.515, 0.545], fakes caught 0.013, reals kept 0.982
- pairing B: accuracy 0.500, AUC 0.523 [0.508, 0.536], fakes caught 0.013, reals kept 0.987

Small differences in the third decimal can come from GPU arithmetic; a difference larger than 0.005 means something
is not identical to 07b (tell me before continuing).

In [ ]:
EXPECTED_07B = {   # Phase 2, SD15_T2I, original images
    "A: DiffusionFace reals": dict(acc=0.498, auc=0.530, auc_lo=0.515, auc_hi=0.545, fake_recall=0.013, real_recall=0.982),
    "B: FFHQ reals":          dict(acc=0.500, auc=0.523, auc_lo=0.508, auc_hi=0.536, fake_recall=0.013, real_recall=0.987),
}

PAIRINGS = {"A: DiffusionFace reals": real_paths}
if USE_FFHQ_REALS and ffhq_real_paths:
    PAIRINGS["B: FFHQ reals"] = ffhq_real_paths

if not os.path.exists(RGB_CKPT):
    print("Phase 2 checkpoint not found - skipping the reproduction check.")
else:
    rgb_model, rgb_cmap = load_resnet(RGB_CKPT)
    for pairing, reals in PAIRINGS.items():
        n = min(len(reals), len(fake_paths["SD15_T2I"]), n_eval)
        items = make_items(reals, fake_paths["SD15_T2I"], n)
        p_real, y_real = score_items(rgb_model, rgb_cmap["real"], items, RGB_TRANSFORM, jpeg_q=None)
        m = metrics_from_scores(p_real, y_real)
        exp = EXPECTED_07B[pairing]
        max_diff = max(abs(m[k] - exp[k]) for k in exp)
        print(f"{pairing:<24} acc {m['acc']:.3f} | AUC {m['auc']:.3f} [{m['auc_lo']:.3f}, {m['auc_hi']:.3f}] | "
              f"fakes caught {m['fake_recall']:.3f} | reals kept {m['real_recall']:.3f}")
        print(f"{'':<24} largest difference from 07b: {max_diff:.4f} -> {'MATCH' if max_diff <= 0.005 else 'DIFFERENT - do not continue'}")
    del rgb_model
    torch.cuda.empty_cache()

A: DiffusionFace reals   acc 0.498 | AUC 0.530 [0.515, 0.545] | fakes caught 0.013 | reals kept 0.982
                         largest difference from 07b: 0.0005 -> MATCH
B: FFHQ reals            acc 0.500 | AUC 0.523 [0.508, 0.536] | fakes caught 0.013 | reals kept 0.987
                         largest difference from 07b: 0.0005 -> MATCH


### Step 10 — Evaluate the Phase 8 frequency model

One model x two pairings x two fake sets x two image variants (original, JPEG-75 re-encoded) = 8 evaluations of
3,000 real + 3,000 fake images. Results are printed and saved to Drive.

In [ ]:
import pandas as pd

VARIANTS = {"original": None, "jpeg75": 75}

freq_model, freq_cmap = load_resnet(FREQ_CKPT)
real_idx = freq_cmap["real"]
print("Frequency model class_to_idx:", freq_cmap)

rows = []
for pairing, reals in PAIRINGS.items():
    for fake_key in FAKE_KEYS:
        n = min(len(reals), len(fake_paths[fake_key]), n_eval)
        items = make_items(reals, fake_paths[fake_key], n)
        for vname, q in VARIANTS.items():
            p_real, y_real = score_items(freq_model, real_idx, items, FREQ_TRANSFORM, jpeg_q=q)
            m = metrics_from_scores(p_real, y_real)
            rows.append(dict(model="Phase 8 (frequency)", pairing=pairing, fake_set=fake_key,
                             variant=vname, n_per_class=n, **m))
            print(f"{pairing:<24} {fake_key:<9} {vname:<9} "
                  f"acc {m['acc']:.3f}  AUC {m['auc']:.3f} [{m['auc_lo']:.3f}, {m['auc_hi']:.3f}]  "
                  f"fakes caught {m['fake_recall']:.3f}  reals kept {m['real_recall']:.3f}")

results = pd.DataFrame(rows)
out_csv = f"{CHECKPOINT_DIR}/diffusionface_frequency_results.csv"
results.to_csv(out_csv, index=False)
print("\nSaved:", out_csv)
results.round(3)

Frequency model class_to_idx: {'fake': 0, 'real': 1}
A: DiffusionFace reals   SD15_T2I  original  acc 0.482  AUC 0.319 [0.305, 0.334]  fakes caught 0.005  reals kept 0.960
A: DiffusionFace reals   SD15_T2I  jpeg75    acc 0.453  AUC 0.365 [0.352, 0.380]  fakes caught 0.100  reals kept 0.807
A: DiffusionFace reals   SD21_T2I  original  acc 0.486  AUC 0.290 [0.277, 0.302]  fakes caught 0.013  reals kept 0.960
A: DiffusionFace reals   SD21_T2I  jpeg75    acc 0.482  AUC 0.432 [0.418, 0.447]  fakes caught 0.156  reals kept 0.807
B: FFHQ reals            SD15_T2I  original  acc 0.420  AUC 0.121 [0.112, 0.130]  fakes caught 0.005  reals kept 0.834
B: FFHQ reals            SD15_T2I  jpeg75    acc 0.462  AUC 0.412 [0.398, 0.426]  fakes caught 0.100  reals kept 0.824
B: FFHQ reals            SD21_T2I  original  acc 0.424  AUC 0.122 [0.113, 0.130]  fakes caught 0.013  reals kept 0.834
B: FFHQ reals            SD21_T2I  jpeg75    acc 0.490  AUC 0.477 [0.462, 0.491]  fakes caught 0.156  reals kept 0

,model,pairing,fake_set,variant,n_per_class,acc,auc,auc_lo,auc_hi,fake_recall,real_recall,bal_acc
0,Phase 8 (frequency),A: DiffusionFace reals,SD15_T2I,original,3000,0.482,0.319,0.305,0.334,0.005,0.960,0.482
1,Phase 8 (frequency),A: DiffusionFace reals,SD15_T2I,jpeg75,3000,0.453,0.365,0.352,0.380,0.100,0.807,0.453
2,Phase 8 (frequency),A: DiffusionFace reals,SD21_T2I,original,3000,0.486,0.290,0.277,0.302,0.013,0.960,0.486
3,Phase 8 (frequency),A: DiffusionFace reals,SD21_T2I,jpeg75,3000,0.482,0.432,0.418,0.447,0.156,0.807,0.482
4,Phase 8 (frequency),B: FFHQ reals,SD15_T2I,original,3000,0.420,0.121,0.112,0.130,0.005,0.834,0.420
5,Phase 8 (frequency),B: FFHQ reals,SD15_T2I,jpeg75,3000,0.462,0.412,0.398,0.426,0.100,0.824,0.462
6,Phase 8 (frequency),B: FFHQ reals,SD21_T2I,original,3000,0.424,0.122,0.113,0.130,0.013,0.834,0.424
7,Phase 8 (frequency),B: FFHQ reals,SD21_T2I,jpeg75,3000,0.490,0.477,0.462,0.491,0.156,0.824,0.490


### Step 11 — Compact summary tables

In [ ]:
summary = results.copy()
summary["AUC [95% CI]"] = summary.apply(lambda r: f"{r['auc']:.3f} [{r['auc_lo']:.3f}, {r['auc_hi']:.3f}]", axis=1)
auc_table = summary.pivot_table(index=["pairing", "fake_set"], columns="variant", values="AUC [95% CI]", aggfunc="first")
print("AUC with 95% bootstrap interval:")
display(auc_table)

acc_table = results.pivot_table(index=["pairing", "fake_set"], columns="variant", values=["acc", "fake_recall", "real_recall"])
print("\nAccuracy, fakes caught, reals kept:")
display(acc_table.round(3))

print("\nReference from 07b (AUC over the same 16 configurations): Phase 2 RGB 0.498-0.530, Phase 6 RGB 0.444-0.504")
print("AUC range of the frequency model here: "
      f"{results['auc'].min():.3f}-{results['auc'].max():.3f}")

AUC with 95% bootstrap interval:


variant                                        jpeg75              original
pairing                fake_set                                            
A: DiffusionFace reals SD15_T2I  0.365 [0.352, 0.380]  0.319 [0.305, 0.334]
                       SD21_T2I  0.432 [0.418, 0.447]  0.290 [0.277, 0.302]
B: FFHQ reals          SD15_T2I  0.412 [0.398, 0.426]  0.121 [0.112, 0.130]
                       SD21_T2I  0.477 [0.462, 0.491]  0.122 [0.113, 0.130]


Accuracy, fakes caught, reals kept:


acc          fake_recall           \
variant                         jpeg75 original      jpeg75 original   
pairing                fake_set                                        
A: DiffusionFace reals SD15_T2I  0.453    0.482       0.100    0.005   
                       SD21_T2I  0.482    0.486       0.156    0.013   
B: FFHQ reals          SD15_T2I  0.462    0.420       0.100    0.005   
                       SD21_T2I  0.490    0.424       0.156    0.013   

                                real_recall           
variant                              jpeg75 original  
pairing                fake_set                       
A: DiffusionFace reals SD15_T2I       0.807    0.960  
                       SD21_T2I       0.807    0.960  
B: FFHQ reals          SD15_T2I       0.824    0.834  
                       SD21_T2I       0.824    0.834


Reference from 07b (AUC over the same 16 configurations): Phase 2 RGB 0.498-0.530, Phase 6 RGB 0.444-0.504
AUC range of the frequency model here: 0.121-0.477


## How to read the results (do not draw conclusions yet)

- **AUC around 0.5**: no usable signal; **below 0.5**: the model ranks fakes as more real than reals; **near 1.0**: the
  model detects this diffusion family.
- **Pairing A vs B**: a higher AUC with FFHQ reals (B) than with DiffusionFace reals (A) means part of any failure comes
  from unfamiliar real photos, not only unfamiliar fakes.
- **original vs jpeg75**: a frequency model may react strongly to JPEG re-encoding, because compression changes the
  spectrum; a large change suggests the score depends on compression and not only on content.
- **Accuracy** is meaningful here because every evaluation has the same number of real and fake images.
- Compare with Phase 8 on the original diffusion set (57.60% accuracy, AUC 0.3027, imbalanced, format confound): this
  is the clean, balanced version of the same test.

**What to send back:** the Step 5 output, the Step 9 output (MATCH / DIFFERENT) and the Step 10 printout (or the CSV).
Nothing in this notebook modifies Phase 8 or any checkpoint.